In [ ]:
%load_ext autoreload
%autoreload 2

import json
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from src.models import step_SIRM, step_SIRT, step_SIRV
from src.sweep import sweep_pol_homophily, sweep_extra_infections, trajectories_at_corners#, run_sweep_multiple_beta
from src.plotting import plot_pol_homophily, plot_extra_infections, plot_I_band
from src.bootstrap import *
from src.sweep import sweep_point_vs_baseline
from src.contact import pol_mean_to_ab, beta_populations

from src.loader import LOAD_parameters, LOAD_plots_specification, LOAD_sweep_config

In [ ]:
mus, taus, xis, PARAMS, rect_coords_M, rect_coords_T, rect_coords_V, simulated_days = LOAD_parameters()
my_map, Lx, Ly, contour_colors = LOAD_plots_specification()
NP, NB, PS, betas, pol_vals, h_vals, build_state0_3, build_state0_4, beh_M, beh_T, beh_V = LOAD_sweep_config(PARAMS)

def run_sweep_multiple_beta(step_fn, build_state0, beh, n_steps, betas):
    results = []
    for b in betas:
        params = {**PARAMS, 'beta_M': b}
        # n_steps should match simulated_days / dT
        final = sweep_pol_homophily(
            step_fn, build_state0, beh,
            pol_vals, h_vals, params, n_steps,
            mean=0.5, n_groups=PS,
        )
        results.append(final)
        print(f"  beta_M = {b} done")
    return results

# Things for Figure 1:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Your custom colorscheme
colors = ['#8e0152', '#c51b7d', '#de77ae', '#f1b6da', '#fde0ef', 
          '#f7f7f7', '#e6f5d0', '#b8e186', '#7fbc41', '#4d9221', '#276419']

# Create a custom colormap using your colors
cmap_name = 'custom_diverging'
custom_cmap = LinearSegmentedColormap.from_list(cmap_name, colors)

# Create a figure and axis
fig, ax = plt.subplots(figsize=(10, 1))

# Create data for the colorbar
gradient = np.linspace(0, 1, 256)
gradient = np.vstack((gradient, gradient))

# Display the colorbar
ax.imshow(gradient, aspect='auto', cmap=custom_cmap)

# Remove ticks from the plot
ax.set_yticks([])
ax.set_xticks(np.linspace(0, 256, 3))
ax.set_xticklabels([])


# Optional: Add a border to the colorbar
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_color('black')
    spine.set_linewidth(0.5)



fig.savefig('figures/Fig_1/custom_colorbar.pdf', dpi=300, bbox_inches='tight')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import beta
from matplotlib.collections import LineCollection
def plot_beta_with_gradient(alpha, beta_param, Nbins=100):
    fig, ax = plt.subplots(figsize=(Lx/3, Ly/5))
    
    x = np.linspace(1/Nbins/2, 1-1/Nbins/2, Nbins)
    y = beta.pdf(x, alpha, beta_param)

    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    lc = LineCollection(segments, cmap=custom_cmap, norm=plt.Normalize(0, 1))
    lc.set_array(x[:-1])
    lc.set_linewidth(3)
    
    line = ax.add_collection(lc)
    ax.plot(x, y, '--', color='black', linewidth=1)
       
    ax.set_xlim(-0.01, 1.01)
    ax.set_ylim(-0.05, 5.05)
    # remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    

    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_yticks([])
    ax.set_yticklabels([])
    fig.patch.set_visible(False)
    plt.show()



    return fig, ax

fig, ax = plot_beta_with_gradient(5, 5, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_sym_1.pdf', dpi=300, bbox_inches='tight')
fig, ax = plot_beta_with_gradient(1, 1, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_sym_2.pdf', dpi=300, bbox_inches='tight')
fig, ax = plot_beta_with_gradient(0.1, 0.1, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_sym_3.pdf', dpi=300, bbox_inches='tight')

fig, ax = plot_beta_with_gradient(5, 0.1, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_asym_1.pdf', dpi=300, bbox_inches='tight')
fig, ax = plot_beta_with_gradient(1, 1, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_asym_2.pdf', dpi=300, bbox_inches='tight')
fig, ax = plot_beta_with_gradient(0.1, 5, Nbins=1000)  # Symmetric
fig.savefig('figures/Fig_1/beta_asym_3.pdf', dpi=300, bbox_inches='tight')

import numpy as np
import matplotlib.pyplot as plt
import jax.numpy as jnp
from typing import List, Optional, Tuple

def plot_contact_matrices(h_values: List[float], 
                         n_groups: int = 5,
                         figsize: Tuple[int, int] = (15, 3),
                         cmap: str = "Blues",
                         save_path: Optional[str] = None,
                         text_flag = True, vmin = 0, vmax = 3) -> plt.Figure:
    """
    Plot contact matrices for different values of homophilic tendency (h).
    
    Args:
        h_values: List of homophilic tendency values to visualize
        n_groups: Number of population groups (matrix size will be n_groups x n_groups)
        figsize: Figure size (width, height)
        cmap: Colormap to use
        save_path: Path to save the figure (if None, figure is not saved)
        
    Returns:
        Matplotlib figure object
    """
    
    # Create a figure with subplots for each h value
    n_plots = len(h_values)
    fig, axes = plt.subplots(1, n_plots, figsize=figsize)
    
    # If only one h value, make axes iterable
    if n_plots == 1:
        axes = [axes]
    
    # Create an equal population distribution
    pop = jnp.ones(n_groups)*0.2
    
    # For each h value, create and plot the contact matrix
    for i, h in enumerate(axes):
        # Create contact matrix
        h_val = h_values[i]
        C = create_contact_matrix(n_groups, h_val, pop)
        C = np.flipud(C)
        # Plot as heatmap
        im = axes[i].imshow(C, cmap=cmap, vmin=vmin, vmax=vmax, extent=[-0.5, n_groups-0.5, -0.5, n_groups-0.5])
        
        if text_flag:
            # Add text annotations to each cell
            for row in range(n_groups):
                for col in range(n_groups):
                    value = np.round(C[row, col],1)
                    # Add text with value to each cell
                    axes[i].text(col, row, f"{value:.1f}", ha="center", va="center", 
                            color="black" if value < 1.5 else "white", fontsize=8)
        
        axes[i].set_xticklabels([])
        axes[i].set_yticklabels([])        
        axes[i].grid(False)
        
        # Add h value as title
        #axes[i].set_title(f"h = {h_val}")
    
    fig.patch.set_visible(False)
    
    # add a grid to separate pixels in the heatmap
    for ax in axes:
        ax.set_xticks(np.arange(0.5, n_groups, 1), minor=True)
        ax.set_yticks(np.arange(0.5, n_groups, 1), minor=True)
        ax.grid(which='minor', color='black', linestyle='-', linewidth=0.5)
        ax.tick_params(which="minor", bottom=False, left=False)


    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return fig

h_values = [0]  # Different homophilic tendency values
fig = plot_contact_matrices(h_values, n_groups=5, save_path='figures/Fig_1/contact_matrices0.pdf', text_flag=False)
plt.show()

h_values = [2]  # Different homophilic tendency values
fig = plot_contact_matrices(h_values, n_groups=5, save_path='figures/Fig_1/contact_matrices2.pdf', text_flag=False)
plt.show()

h_values = [4]  # Different homophilic tendency values
fig = plot_contact_matrices(h_values, n_groups=5, save_path='figures/Fig_1/contact_matrices4.pdf', text_flag=False)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

def make_colorbar(cmap, vmin=0, vmax=1):
    """Generate only a colorbar with specified colormap
    
    Args:
        cmap: colormap object or string name
    """
    
    fig = plt.figure(figsize=(1 / 25.4, 100 / 25.4))
    ax = fig.add_axes([0.05, 0.4, 0.9, 0.2])
    
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    
    if isinstance(cmap, str):
        cmap = mpl.colormaps[cmap]
    
    cb = mpl.colorbar.ColorbarBase(ax, cmap=cmap, norm=norm, orientation='vertical')
    
    ax.set_yticks([0,0.5,1])
    ax.set_yticklabels([])

    ax.tick_params(width=0.5)
    fig.patch.set_visible(False)
    # reduce thickness of colorbar border
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    return fig

fig_cb = make_colorbar("Blues", vmin=0, vmax=1)
fig_cb.savefig('figures/Fig_1/colorbar_contact_matrix.pdf', dpi=300, bbox_inches='tight')

In [ ]:
print(beh_M)
print(beh_T)
print(beh_V)

In [ ]:
print("### SIR-M ###")
res_list_M_2 = run_sweep_multiple_beta(step_SIRM, build_state0_3, beh_M, simulated_days*4, betas)

print("### SIR-T ###")
res_list_T_2 = run_sweep_multiple_beta(step_SIRT, build_state0_3, beh_T, simulated_days*4, betas)

print("### SIR-V ###")
res_list_V_2 = run_sweep_multiple_beta(step_SIRV, build_state0_4, beh_V, simulated_days*4, betas)

In [ ]:
final_params = dict(
    fig_size=(1.83, 1.73),
    xticks=[0, 0.5, 1.0],
    yticks=[0, 3, 6],
    xlim=[0, 1],
    ylim=[0, 6],
)

path_1_M_L = "figures/Fig_2/masks_L_08.pdf"
path_1_M_M = "figures/Fig_2/masks_M_08.pdf"
path_2_M_H = "figures/Fig_2/masks_H_08.pdf"

pol_np = np.asarray(pol_vals)
h_np = np.asarray(h_vals)

fig_R_M_1_L = plot_pol_homophily(
    res_list_M_2[0], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    save_path=path_1_M_L,
    **final_params,
)

fig_R_M_1_H = plot_pol_homophily(
    res_list_M_2[1], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.5],
    contour_colors=contour_colors,
    save_path=path_1_M_M,
    **final_params,
)

fig_R_M_2 = plot_pol_homophily(
    res_list_M_2[2], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.6, 0.8],
    contour_colors=contour_colors,
    save_path=path_2_M_H,
    **final_params,
)

In [ ]:
final_params = dict(
    fig_size=(1.83, 1.73),
    xticks=[0, 0.5, 1.0],
    yticks=[0, 3, 6],
    xlim=[0, 1],
    ylim=[0, 6],
)

path_1_T_L = "figures/Fig_2/tests_L.pdf"
path_1_T_M = "figures/Fig_2/tests_M.pdf"
path_1_T_H = "figures/Fig_2/tests_H.pdf"

pol_np = np.asarray(pol_vals)
h_np = np.asarray(h_vals)

fig_R_M_1_L = plot_pol_homophily(
    res_list_T_2[0], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    save_path=path_1_T_L,
    **final_params,
)

fig_R_M_1_H = plot_pol_homophily(
    res_list_T_2[1], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.4, 0.5, 0.6],
    contour_colors=contour_colors,
    save_path=path_1_T_M,
    **final_params,
)

fig_R_M_2 = plot_pol_homophily(
    res_list_T_2[2], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8, 0.9],
    contour_colors=contour_colors,
    save_path=path_1_T_H,
    **final_params,
)

In [ ]:
final_params = dict(
    fig_size=(1.83, 1.73),
    xticks=[0, 0.5, 1.0],
    yticks=[0, 3, 6],
    xlim=[0, 1],
    ylim=[0, 6],
)

path_1_V_L = "figures/Fig_2/vaccines_L.pdf"
path_1_V_M = "figures/Fig_2/vaccines_M.pdf"
path_1_V_H = "figures/Fig_2/vaccines_H.pdf"

pol_np = np.asarray(pol_vals)
h_np = np.asarray(h_vals)

fig_R_M_1_L = plot_pol_homophily(
    res_list_V_2[0], ['S', 'I', 'R', 'V'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    save_path=path_1_V_L,
    **final_params,
)

fig_R_M_1_H = plot_pol_homophily(
    res_list_V_2[1], ['S', 'I', 'R', 'V'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.4],
    contour_colors=contour_colors,
    save_path=path_1_V_M,
    **final_params,
)

fig_R_M_2 = plot_pol_homophily(
    res_list_V_2[2], ['S', 'I', 'R', 'V'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6,0.7, 0.8],
    contour_colors=contour_colors,
    save_path=path_1_V_H,
    **final_params,
)

In [ ]:

print("### SIR-M ###")
mean_M = mus["mean"][1]
res_list_M = []
for b in betas:
    params = {**PARAMS, 'beta_M': b}
    final = sweep_pol_homophily(
        step_SIRM, build_state0_3, beh_M,
        pol_vals, h_vals, params, n_steps=simulated_days*4,
        mean=mean_M, n_groups=PS,
    )
    res_list_M.append(final)
    print(f"  beta_M = {b} done")

print("### SIR-T ###")
mean_T = taus["mean"][1]
res_list_T = []
for b in betas:
    params = {**PARAMS, 'beta_M': b}
    final = sweep_pol_homophily(
        step_SIRT, build_state0_3, beh_T,
        pol_vals, h_vals, params, n_steps=simulated_days*4,
        mean=mean_T, n_groups=PS,
    )
    res_list_T.append(final)
    print(f"  beta_M = {b} done")

print("### SIR-V ###")
mean_V = xis["mean"][1]
res_list_V = []
for b in betas:
    params = {**PARAMS, 'beta_M': b}
    final = sweep_pol_homophily(
        step_SIRV, build_state0_4, beh_V,
        pol_vals, h_vals, params, n_steps=simulated_days*4,
        mean=mean_V, n_groups=PS,
    )
    res_list_V.append(final)
    print(f"  beta_M = {b} done")

In [ ]:
path_1_M_M = "figures/Fig_3/masks_M_08.pdf"

fig_R_M_1_L = plot_pol_homophily(
    res_list_M[0], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    rect_coords=rect_coords_M,
    **final_params,
)

fig_R_M_1_H = plot_pol_homophily(
    res_list_M[1], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    rect_coords=rect_coords_M,
    save_path=path_1_M_M,
    **final_params,
)

fig_R_M_2 = plot_pol_homophily(
    res_list_M[2], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    rect_coords=rect_coords_M,
    **final_params,
)

In [ ]:
path_1_T_M = "figures/Fig_3/tests_M.pdf"


fig_R_M_1_L = plot_pol_homophily(
    res_list_T[0], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    rect_coords=rect_coords_T,
    **final_params,
)

fig_R_M_1_H = plot_pol_homophily(
    res_list_T[1], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.5, 0.6],
    contour_colors=contour_colors,
    rect_coords=rect_coords_T,
    save_path=path_1_T_M,
    **final_params,
)

fig_R_M_2 = plot_pol_homophily(
    res_list_T[2], ['S', 'I', 'R'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    rect_coords=rect_coords_T,
    **final_params,
)

In [ ]:
path_1_V_M = "figures/Fig_3/vaccines_M.pdf"


fig_R_M_1_L = plot_pol_homophily(
    res_list_V[0], ['S', 'I', 'R', 'V'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    rect_coords=rect_coords_V,
    **final_params,
)

fig_R_M_1_H = plot_pol_homophily(
    res_list_V[1], ['S', 'I', 'R', 'V'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    rect_coords=rect_coords_V,
    save_path=path_1_V_M,
    **final_params,
)

fig_R_M_2 = plot_pol_homophily(
    res_list_V[2], ['S', 'I', 'R', 'V'], pol_np, h_np,
    cmap=my_map,
    contour_values=[0.2, 0.4, 0.5, 0.6, 0.8],
    contour_colors=contour_colors,
    rect_coords=rect_coords_V,
    **final_params,
)

In [ ]:
beta_vals = jnp.linspace(0.10, 0.50, 200) 
gamma = PARAMS['recovery_rate']

I_corners_M, I_base_M = sweep_extra_infections(
    step_SIRM, build_state0_3, beh_M, beta_vals, rect_coords_M,
    PARAMS, n_steps=12000, mean=mus['mean'][1], n_groups=PS,
)
I_corners_T, I_base_T = sweep_extra_infections(
    step_SIRT, build_state0_3, beh_T, beta_vals, rect_coords_T,
    PARAMS, n_steps=12000, mean=taus['mean'][1], n_groups=PS,
)
I_corners_V, I_base_V = sweep_extra_infections(
    step_SIRV, build_state0_4, beh_V, beta_vals, rect_coords_V,
    PARAMS, n_steps=12000, mean=xis['mean'][1], n_groups=PS,
)

fig = plot_extra_infections(
    beta_vals, gamma,
    I_corners_M, I_base_M,
    I_corners_T, I_base_T,
    I_corners_V, I_base_V,
    save_path='figures/Fig_4/percentage_increase.pdf',
)

In [ ]:
beta_0 = 0.25   # pick whichever beta_M you want to spotlight

I_corners_T, I_base_T = trajectories_at_corners(
    step_SIRT, build_state0_3, beh_T, rect_coords_T, beta_0,
    PARAMS, n_steps=1000, mean=taus['mean'][1], n_groups=PS,
)

fig = plot_I_band(I_corners_T, I_base_T, dT=PARAMS['dT'],
                  color='#d95f02')

In [ ]:
beta_0 = 0.25   # pick whichever beta_M you want to spotlight

path1 =  "figures/Fig_3/comparison_M.pdf"
path2 =  "figures/Fig_3/comparison_T.pdf"
path3 =  "figures/Fig_3/comparison_V.pdf"



# --- SIR-M ---
I_corners_M, I_base_M = trajectories_at_corners(
    step_SIRM, build_state0_3, beh_M, rect_coords_M, beta_0,
    PARAMS, n_steps=1000, mean=mus['mean'][1], n_groups=PS,
)
fig_M = plot_I_band(I_corners_M, I_base_M, dT=PARAMS['dT'],
                    color='#7570b3', save_path = path1)

# --- SIR-T ---
I_corners_T, I_base_T = trajectories_at_corners(
    step_SIRT, build_state0_3, beh_T, rect_coords_T, beta_0,
    PARAMS, n_steps=1000, mean=taus['mean'][1], n_groups=PS,
)
fig_T = plot_I_band(I_corners_T, I_base_T, dT=PARAMS['dT'],
                    color='#d95f02', save_path = path2)

# --- SIR-V ---
I_corners_V, I_base_V = trajectories_at_corners(
    step_SIRV, build_state0_4, beh_V, rect_coords_V, beta_0,
    PARAMS, n_steps=1000, mean=xis['mean'][1], n_groups=PS,
)
fig_V = plot_I_band(I_corners_V, I_base_V, dT=PARAMS['dT'],
                    color='#1b9e77', save_path = path3)

In [ ]:
df = pd.read_csv("data_homophily.csv")
df = df.dropna()

mask_distribution = extract_behavior_distribution_vectorized(df, "masks")
test_distribution = extract_behavior_distribution_vectorized(df, "testing")
vaccine_distribution = extract_behavior_distribution_vectorized(df, "vacc")

mask_matrix = generate_contact_matrix(df, "masks")
test_matrix = generate_contact_matrix(df, "testing")
vaccine_matrix = generate_contact_matrix(df, "vacc")

In [ ]:
fig, ax = plot_histogram_distribution(mask_distribution,  save_path="figures/Fig_3/masks_distribution.pdf")
fig, ax = plot_histogram_distribution(test_distribution,  save_path="figures/Fig_3/testing_distribution.pdf")
fig, ax = plot_histogram_distribution(vaccine_distribution,  save_path="figures/Fig_3/vaccine_distribution.pdf")

In [ ]:
fig, ax = plot_contact_matrix(mask_matrix, path="figures/Fig_3/mask_contact_matrix.pdf")
fig, ax = plot_contact_matrix(test_matrix, path="figures/Fig_3/testing_contact_matrix.pdf")
fig, ax = plot_contact_matrix(vaccine_matrix, path="figures/Fig_3/vaccine_contact_matrix.pdf")

In [ ]:
# calculate and print the range of the confidence interval:
R_M = bootstrap_mph(df, "masks", n_bootstrap=20000)
R_T = bootstrap_mph(df, "testing", n_bootstrap=20000)
R_V = bootstrap_mph(df, "vacc", n_bootstrap=20000)

print(f"[{np.round(R_M['M_CI'], 2)}, {np.round(R_M['P_CI'], 2)}, {np.round(R_M['H_CI'], 2)}]")
print(f"[{np.round(R_T['M_CI'], 2)}, {np.round(R_T['P_CI'], 2)}, {np.round(R_T['H_CI'], 2)}]")
print(f"[{np.round(R_V['M_CI'], 2)}, {np.round(R_V['P_CI'], 2)}, {np.round(R_V['H_CI'], 2)}]")

# SI things

## SI1

In [ ]:
fig, axes = plot_bootstrap_heatmap(R_M, R_T, R_V, bins = 150, figsize=(7.09, 7))
axes[(0,0)].set_title("Masks", fontsize=12)
axes[(0,1)].set_title("Testing", fontsize=12)
axes[(0,2)].set_title("Vaccines", fontsize=12)
for i in range(3):
    axes[(0,i)].set_xlabel("Mean", fontsize=10)
for i in range(2):
    axes[(i,0)].set_ylabel("Homophily", fontsize=10)
axes[(2,0)].set_ylabel("Mean", fontsize=10)
for i in range(3):
    axes[(2,i)].set_xlabel("Polarization", fontsize=10)


for i in range(3):
    axes[(0,i)].set_xlim(0.4, 0.85)
    axes[(1,i)].set_xlim(0.2, 0.65)
    axes[(2,i)].set_xlim(0.2, 0.65)

    axes[(0,i)].set_xticks([0.4, 0.6, 0.8])
    axes[(1,i)].set_xticks([0.2, 0.4,  0.6])
    axes[(2,i)].set_xticks([0.2, 0.4,  0.6])

for i in range(3):
    axes[(0,i)].set_ylim(1.25,3.75)
    axes[(1,i)].set_ylim(1.25,3.75)
    axes[(2,i)].set_ylim(0.4,0.85)
    axes[(2,i)].set_yticks([0.4, 0.6, 0.8])


fig.savefig("Figures/Fig_SI/Fig_SI_bootstrapping.pdf", bbox_inches='tight')

## SI 2

In [ ]:
import numpy as np
import jax.numpy as jnp
from src.sweep import sweep_pol_homophily
from src.models import step_SIRM, step_SIRT, step_SIRV

# --- grid ---
pol_vals = np.linspace(1e-3, 1.0, NP)
h_vals = np.linspace(0.0, 6.0, NP)
N_vals = [3, 5, 10, 30]
n_steps = 3000

# --- placeholders: fill these in ---
# beh_fn(n_groups) -> per-group behavior array of length n_groups
# (the discretized behavior level for each bin, e.g. midpoints 0..1)
def beh_fn(n_groups):
    return jnp.linspace(0.0, 1.0, n_groups)

# build_state0(pop) -> initial (n_comp, n_groups) state
def build_state0_SIR(pop):
    I0 = pop * 1e-3
    return jnp.stack([pop - I0, I0, jnp.zeros_like(pop)])

def build_state0_SIRV(pop):
    I0 = pop * 1e-3
    return jnp.stack([pop - I0, I0, jnp.zeros_like(pop), jnp.zeros_like(pop)])

params = {'beta_M': 0.25, 'recovery_rate': 0.1, 'dT': 0.25}  # placeholder

models = {
    'SIRM': (step_SIRM, build_state0_SIR),
    'SIRT': (step_SIRT, build_state0_SIR),
    'SIRV': (step_SIRV, build_state0_SIRV),
}

# --- compute ---
for name, (step_fn, build_state0) in models.items():
    for N in N_vals:
        print(f"Running {name} with N={N} groups...")
        beh = beh_fn(N)
        final_state = sweep_pol_homophily(
            step_fn, build_state0, beh, pol_vals, h_vals,
            params, n_steps, mean=0.5, n_groups=N,
        )
        np.save(f"data/N_sweep_{name}_N{N}.npy", np.asarray(final_state))

np.save("data/N_sweep_pol_vals.npy", pol_vals)
np.save("data/N_sweep_h_vals.npy", h_vals)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N_vals = [3, 5, 10, 30]
models = ['SIRM', 'SIRT', 'SIRV']
compartments = {'SIRM': ['S', 'I', 'R'],
                'SIRT': ['S', 'I', 'R'],
                'SIRV': ['S', 'I', 'R', 'V']}

pol_vals = np.load("data/N_sweep_pol_vals.npy")
h_vals = np.load("data/N_sweep_h_vals.npy")


def infection_size(final_state, comp_names):
    S_idx = comp_names.index('S')
    data = 1.0 - final_state[..., S_idx, :].sum(axis=-1)
    if 'V' in comp_names:
        V_idx = comp_names.index('V')
        data = data - final_state[..., V_idx, :].sum(axis=-1)
    return data  # (n_pol, n_h)


fig, axes = plt.subplots(3, 4, figsize=(7, 5))

for row, name in enumerate(models):
    comp_names = compartments[name]
    for col, N in enumerate(N_vals):
        final_state = np.load(f"data/N_sweep_{name}_N{N}.npy")
        data = infection_size(final_state, comp_names).T  # (n_h, n_pol)

        ax = axes[row, col]
        mesh = ax.pcolormesh(pol_vals, h_vals, data,
                             cmap=my_map, vmin=0, vmax=1)
        ax.contour(pol_vals, h_vals, data, levels=np.linspace(0.1, 0.9, 9),
                   colors='black', linewidths=0.4, alpha=0.5)

        if row == 0:
            ax.set_title(f"N = {N}", fontsize=11)
        if col == 0:
            ax.set_ylabel(name + "\nHomophily", fontsize=10)
            ax.set_yticks([0, 3, 6])
        else:
            ax.set_yticks([0, 3, 6])
            ax.set_yticklabels([])
        if row == 2:
            ax.set_xlabel("Polarization", fontsize=9)
            ax.set_xticks([0, 0.5, 1])
        else:
            ax.set_xticks([0, 0.5, 1])
            ax.set_xticklabels([])

cbar = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Fraction of Infected")
cbar.set_ticks([0, 0.5, 1])

fig.savefig("figures/Fig_SI/dependency_on_NCOMPARTMENTS.pdf", dpi=300, bbox_inches='tight')

## SI 3

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from src.sweep import sweep_pol_homophily
from src.models import step_SIRM

# --- grid ---
pol_vals = np.linspace(1e-3, 1.0-1e-3, 41)
h_vals = np.linspace(0.0, 6.0, 41)
mu_min_vals = [0.0, 0.2, 0.4]
mu_max_vals = [0.6, 0.8, 1.0]
comp_names = ['S', 'I', 'R']
N = 5
n_steps = 3000

# --- placeholders: fill these in ---
def beh_fn(n_groups, mu_min, mu_max):
    return mu_min + (mu_max - mu_min) * jnp.linspace(0.0, 1.0, n_groups)

def build_state0_SIR(pop):
    I0 = pop * 1e-6
    return jnp.stack([pop - I0, I0, jnp.zeros_like(pop)])

params = {'beta_M': 0.25, 'recovery_rate': 0.1, 'dT': 1.0}  # placeholder


def infection_size(final_state, comp_names):
    S_idx = comp_names.index('S')
    data = 1.0 - final_state[..., S_idx, :].sum(axis=-1)
    if 'V' in comp_names:
        V_idx = comp_names.index('V')
        data = data - final_state[..., V_idx, :].sum(axis=-1)
    return data  # (n_pol, n_h)


# --- compute ---
results = {}
for mu_min in mu_min_vals:
    for mu_max in mu_max_vals:
        beh = beh_fn(N, mu_min, mu_max)
        final_state = sweep_pol_homophily(
            step_SIRM, build_state0_SIR, beh, pol_vals, h_vals,
            params, n_steps, mean=0.5, n_groups=N,
        )
        results[(mu_min, mu_max)] = infection_size(final_state, comp_names)


In [ ]:

# --- plot ---
fig, axes = plt.subplots(3, 3, figsize=(7.09, 6))

for row, mu_min in enumerate(mu_min_vals):
    for col, mu_max in enumerate(mu_max_vals):
        data = results[(mu_min, mu_max)].T  # (n_h, n_pol)

        ax = axes[row, col]
        mesh = ax.pcolormesh(pol_vals, h_vals, data,
                             cmap=my_map, vmin=0, vmax=1)
        ax.contour(pol_vals, h_vals, data, levels=np.linspace(0.1, 0.9, 9),
                   colors='black', linewidths=0.4, alpha=0.5)

        if col == 0:
            ax.set_ylabel(f"$\\mu_{{min}}$ = {mu_min}\nHomophily", fontsize=9)
        ax.set_yticks([0, 3, 6])
        if col != 0:
            ax.set_yticklabels([])
        if row == 0:
            ax.set_title(f"$\\mu_{{max}}$ = {mu_max}", fontsize=11)
        ax.set_xticks([0, 0.5, 1])
        if row == 2:
            ax.set_xlabel("Polarization", fontsize=9)
        else:
            ax.set_xticklabels([])

cbar = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Fraction of Infected")
cbar.set_ticks([0, 0.5, 1])

plt.savefig("figures/Fig_SI/MIN-MAX_SIRM.pdf", dpi=300, bbox_inches='tight')

## SI4

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from src.sweep import sweep_pol_homophily
from src.models import step_SIRT

# --- grid ---
pol_vals = np.linspace(1e-3, 1.0-1e-3, 41)
h_vals = np.linspace(0.0, 6.0, 41)
tau_min_vals = [0.0, 0.025, 0.05]
tau_max_vals = [0.1, 0.2, 0.3]
comp_names = ['S', 'I', 'R']
N = 5
n_steps = 3000

# --- placeholders: fill these in ---
def beh_fn(n_groups, tau_min, tau_max):
    return tau_min + (tau_max - tau_min) * jnp.linspace(0.0, 1.0, n_groups)

def build_state0_SIR(pop):
    I0 = pop * 1e-6
    return jnp.stack([pop - I0, I0, jnp.zeros_like(pop)])

params = {'beta_M': 0.25, 'recovery_rate': 0.1, 'dT': 1.0}  # placeholder


def infection_size(final_state, comp_names):
    S_idx = comp_names.index('S')
    data = 1.0 - final_state[..., S_idx, :].sum(axis=-1)
    if 'V' in comp_names:
        V_idx = comp_names.index('V')
        data = data - final_state[..., V_idx, :].sum(axis=-1)
    return data  # (n_pol, n_h)


# --- compute ---
results = {}
for tau_min in tau_min_vals:
    for tau_max in tau_max_vals:
        beh = beh_fn(N, tau_min, tau_max)
        final_state = sweep_pol_homophily(
            step_SIRT, build_state0_SIR, beh, pol_vals, h_vals,
            params, n_steps, mean=0.5, n_groups=N,
        )
        results[(tau_min, tau_max)] = infection_size(final_state, comp_names)

# --- plot ---
fig, axes = plt.subplots(3, 3, figsize=(7.09, 6))

for row, tau_min in enumerate(tau_min_vals):
    for col, tau_max in enumerate(tau_max_vals):
        data = results[(tau_min, tau_max)].T  # (n_h, n_pol)

        ax = axes[row, col]
        mesh = ax.pcolormesh(pol_vals, h_vals, data,
                             cmap=my_map, vmin=0, vmax=1)
        ax.contour(pol_vals, h_vals, data, levels=np.linspace(0.1, 0.9, 9),
                   colors='black', linewidths=0.4, alpha=0.5)

        if col == 0:
            ax.set_ylabel(f"$\\tau_{{min}}$ = {tau_min}\nHomophily", fontsize=9)
        ax.set_yticks([0, 3, 6])
        if col != 0:
            ax.set_yticklabels([])
        if row == 0:
            ax.set_title(f"$\\tau_{{max}}$ = {tau_max}", fontsize=11)
        ax.set_xticks([0, 0.5, 1])
        if row == 2:
            ax.set_xlabel("Polarization", fontsize=9)
        else:
            ax.set_xticklabels([])

cbar = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Fraction of Infected")
cbar.set_ticks([0, 0.5, 1])


## SI 5

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from src.sweep import sweep_pol_homophily
from src.models import step_SIRV

# --- grid ---
pol_vals = np.linspace(1e-3, 1.0-1e-3, 41)
h_vals = np.linspace(0.0, 6.0, 41)
xi_min_vals = [0.0, 0.002, 0.004]
xi_max_vals = [0.005, 0.01, 0.015]
comp_names = ['S', 'I', 'R', 'V']
N = 5
n_steps = 3000

# --- placeholders: fill these in ---
def beh_fn(n_groups, xi_min, xi_max):
    return xi_min + (xi_max - xi_min) * jnp.linspace(0.0, 1.0, n_groups)

def build_state0_SIRV(pop):
    I0 = pop * 1e-6
    Z = jnp.zeros_like(pop)
    return jnp.stack([pop - I0, I0, Z, Z])

params = {'beta_M': 0.25, 'recovery_rate': 0.1, 'dT': 1.0}  # placeholder


def infection_size(final_state, comp_names):
    S_idx = comp_names.index('S')
    data = 1.0 - final_state[..., S_idx, :].sum(axis=-1)
    if 'V' in comp_names:
        V_idx = comp_names.index('V')
        data = data - final_state[..., V_idx, :].sum(axis=-1)
    return data  # (n_pol, n_h)


# --- compute ---
results = {}
for xi_min in xi_min_vals:
    for xi_max in xi_max_vals:
        beh = beh_fn(N, xi_min, xi_max)
        final_state = sweep_pol_homophily(
            step_SIRV, build_state0_SIRV, beh, pol_vals, h_vals,
            params, n_steps, mean=0.5, n_groups=N,
        )
        results[(xi_min, xi_max)] = infection_size(final_state, comp_names)


In [ ]:

# --- plot ---
fig, axes = plt.subplots(3, 3, figsize=(7.09, 6))

for row, xi_min in enumerate(xi_min_vals):
    for col, xi_max in enumerate(xi_max_vals):
        data = results[(xi_min, xi_max)].T  # (n_h, n_pol)

        ax = axes[row, col]
        mesh = ax.pcolormesh(pol_vals, h_vals, data,
                             cmap=my_map, vmin=0, vmax=1)
        ax.contour(pol_vals, h_vals, data, levels=np.linspace(0.1, 0.9, 9),
                   colors='black', linewidths=0.4, alpha=0.5)

        if col == 0:
            ax.set_ylabel(f"$\\xi_{{min}}$ = {xi_min}\nHomophily", fontsize=9)
        ax.set_yticks([0, 3, 6])
        if col != 0:
            ax.set_yticklabels([])
        if row == 0:
            ax.set_title(f"$\\xi_{{max}}$ = {xi_max}", fontsize=11)
        ax.set_xticks([0, 0.5, 1])
        if row == 2:
            ax.set_xlabel("Polarization", fontsize=9)
        else:
            ax.set_xticklabels([])

cbar = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Fraction of Infected")
cbar.set_ticks([0, 0.5, 1])

plt.savefig("figures/Fig_SI/MIN-MAX_SIRV.pdf.pdf", dpi=300, bbox_inches='tight')

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from src.sweep import sweep_pol_homophily
from src.models import step_SIRV

# --- grid ---
pol_vals = np.linspace(1e-3, 1.0-1e-3, 41)
h_vals = np.linspace(0.0, 6.0, 41)
I0_vals = [1e-9, 1e-6, 1e-4, 1e-3]
xi_max_vals = [0.005, 0.01, 0.015]
xi_min = 0.0
comp_names = ['S', 'I', 'R', 'V']
N = 5
n_steps = 3000

# --- placeholders: fill these in ---
def beh_fn(n_groups, xi_min, xi_max):
    return xi_min + (xi_max - xi_min) * jnp.linspace(0.0, 1.0, n_groups)

def build_state0_SIRV(pop, I0):
    I = pop * I0
    Z = jnp.zeros_like(pop)
    return jnp.stack([pop - I, I, Z, Z])

params = {'beta_M': 0.25, 'recovery_rate': 0.1, 'dT': 1.0}  # placeholder


def infection_size(final_state, comp_names):
    S_idx = comp_names.index('S')
    data = 1.0 - final_state[..., S_idx, :].sum(axis=-1)
    if 'V' in comp_names:
        V_idx = comp_names.index('V')
        data = data - final_state[..., V_idx, :].sum(axis=-1)
    return data  # (n_pol, n_h)


# --- compute ---
results = {}
for I0 in I0_vals:
    state0_fn = lambda pop, I0=I0: build_state0_SIRV(pop, I0)
    for xi_max in xi_max_vals:
        beh = beh_fn(N, xi_min, xi_max)
        final_state = sweep_pol_homophily(
            step_SIRV, state0_fn, beh, pol_vals, h_vals,
            params, n_steps, mean=0.5, n_groups=N,
        )
        results[(I0, xi_max)] = infection_size(final_state, comp_names)

# --- plot ---
fig, axes = plt.subplots(4, 3, figsize=(7.09, 8))

for row, I0 in enumerate(I0_vals):
    for col, xi_max in enumerate(xi_max_vals):
        data = results[(I0, xi_max)].T  # (n_h, n_pol)

        ax = axes[row, col]
        mesh = ax.pcolormesh(pol_vals, h_vals, data,
                             cmap=my_map, vmin=0, vmax=1)
        ax.contour(pol_vals, h_vals, data, levels=np.linspace(0.1, 0.9, 9),
                   colors='black', linewidths=0.4, alpha=0.5)

        if col == 0:
            exp = int(round(np.log10(I0)))
            ax.set_ylabel(f"$I_0 = 10^{{{exp}}}$\nHomophily", fontsize=9)
        ax.set_yticks([0, 3, 6])
        if col != 0:
            ax.set_yticklabels([])
        if row == 0:
            ax.set_title(f"$\\xi_{{max}}$ = {xi_max}", fontsize=11)
        ax.set_xticks([0, 0.5, 1])
        if row == 3:
            ax.set_xlabel("Polarization", fontsize=9)
        else:
            ax.set_xticklabels([])

cbar = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Fraction of Infected")
cbar.set_ticks([0, 0.5, 1])

plt.savefig("figures/Fig_SI/SIRV_I0.pdf", dpi=300, bbox_inches='tight')

In [ ]:
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib import cm
from src.sweep import sweep_pol_mean
from src.contact import pol_mean_to_ab
from src.models import step_SIRM, step_SIRT, step_SIRV

# --- grid ---
pol_vals = np.linspace(1e-3, 1.0-1e-3, 81)
mean_vals = np.linspace(1e-3, 1.0-1e-3, 81)
h_vals = [0.0, 3.0, 6.0]
N = 5
n_steps = 3000

# --- per-model setup (reused from earlier scripts) ---
def build_state0_SIR(pop):
    I0 = pop * 1e-6
    Z = jnp.zeros_like(pop)
    return jnp.stack([pop - I0, I0, Z])

def build_state0_SIRV(pop):
    I0 = pop * 1e-6
    Z = jnp.zeros_like(pop)
    return jnp.stack([pop - I0, I0, Z, Z])

params = {'beta_M': 0.25, 'recovery_rate': 0.1, 'dT': 1.0}  # placeholder

models = {
    'SIRM': (step_SIRM, build_state0_SIR, ['S', 'I', 'R'],
             jnp.linspace(0.0, 0.8, N)),       # mask: 0 to theta_max
    'SIRT': (step_SIRT, build_state0_SIR, ['S', 'I', 'R'],
             jnp.linspace(0.0, 0.198, N)),     # testing: 0 to tau_max
    'SIRV': (step_SIRV, build_state0_SIRV, ['S', 'I', 'R', 'V'],
             jnp.linspace(0.0, 0.01, N)),      # vaccination: 0 to xi_max
}


def infection_size(final_state, comp_names):
    S_idx = comp_names.index('S')
    data = 1.0 - final_state[..., S_idx, :].sum(axis=-1)
    if 'V' in comp_names:
        V_idx = comp_names.index('V')
        data = data - final_state[..., V_idx, :].sum(axis=-1)
    return data  # (n_pol, n_mean)


def feasibility_mask(pol_vals, mean_vals):
    """True where the Beta (a, b) is infeasible (a<=0 or b<=0)."""
    mask = np.zeros((len(pol_vals), len(mean_vals)), dtype=bool)
    for i, pol in enumerate(pol_vals):
        for j, mean in enumerate(mean_vals):
            a, b = pol_mean_to_ab(float(pol), float(mean))
            mask[i, j] = (a <= 0) or (b <= 0)
    return mask


mask = feasibility_mask(pol_vals, mean_vals)  # (n_pol, n_mean)

# --- compute ---
results = {}
for name, (step_fn, build_state0, comp_names, beh) in models.items():
    for h in h_vals:
        final_state = sweep_pol_mean(
            step_fn, build_state0, beh, pol_vals, mean_vals,
            params, n_steps, h=h, n_groups=N,
        )
        data = infection_size(final_state, comp_names)
        data = np.where(mask, np.nan, data)
        results[(name, h)] = data


In [ ]:
# --- plot ---
cmap = cm.get_cmap('hot').copy()
cmap.set_bad('0.6')   # gray for infeasible region

fig, axes = plt.subplots(3, 3, figsize=(7.09, 6))

for row, name in enumerate(models):
    for col, h in enumerate(h_vals):
        data = results[(name, h)].T  # (n_mean, n_pol)

        ax = axes[row, col]
        mesh = ax.pcolormesh(pol_vals, mean_vals, data,
                             cmap=my_map, vmin=0, vmax=1)
        ax.contour(pol_vals, mean_vals, data,
                   levels=np.linspace(0.0, 1, 10),
                   colors='black', linewidths=0.3, alpha=0.5)

        if col == 0:
            ax.set_ylabel(f"{name}\nMean", fontsize=9)
        ax.set_yticks([0, 0.5, 1])
        if col != 0:
            ax.set_yticklabels([])
        if row == 0:
            ax.set_title(f"Homophily = {h:g}", fontsize=11)
        ax.set_xticks([0, 0.5, 1])
        if row == 2:
            ax.set_xlabel("Polarization", fontsize=9)
        else:
            ax.set_xticklabels([])

cbar = fig.colorbar(mesh, ax=axes, fraction=0.025, pad=0.02)
cbar.set_label("Fraction of Infected")
cbar.set_ticks([0, 0.5, 1])

fig.savefig("figures/Fig_SI/mean_comparison.pdf", dpi=300, bbox_inches='tight')

In [ ]:

# --- axes of the figure ---
pol_vals = [0.25, 0.33, 0.50]      # columns
hom_vals = [1.0, 3.0, 5.0]         # rows
mean_vals = [0.3, 0.5, 0.7]        # shades within each panel

beta_vals = jnp.linspace(0.10, 0.50, 200)
gamma = 0.1
R0s = np.asarray(beta_vals) / gamma
N = 5
n_steps = 3000

# --- per-model setup (reused from earlier scripts) ---
def build_state0_3(pop):
    I0 = pop * 1e-6
    Z = jnp.zeros_like(pop)
    return jnp.stack([pop - I0, I0, Z])

def build_state0_4(pop):
    I0 = pop * 1e-6
    Z = jnp.zeros_like(pop)
    return jnp.stack([pop - I0, I0, Z, Z])

PARAMS = {'beta_M': 0.25, 'recovery_rate': gamma, 'dT': 0.25}  # placeholder

beh_M = jnp.linspace(0.0, 0.8, N)       # mask: 0 to theta_max
beh_T = jnp.linspace(0.0, 0.198, N)     # testing: 0 to tau_max
beh_V = jnp.linspace(0.0, 0.01, N)      # vaccination: 0 to xi_max

models = {
    'M': (step_SIRM, build_state0_3, beh_M),
    'T': (step_SIRT, build_state0_3, beh_T),
    'V': (step_SIRV, build_state0_4, beh_V),
}

# --- compute structural bias: (I_point - I_baseline)*100 ---
# results[(model, k, j, m)] = array over R0s
results = {}
for name, (step_fn, build_state0, beh) in models.items():
    for k, pol in enumerate(pol_vals):
        for j, h in enumerate(hom_vals):
            for m, mean in enumerate(mean_vals):
                I_pt, I_base = sweep_point_vs_baseline(
                    step_fn, build_state0, beh, beta_vals, pol, h,
                    PARAMS, n_steps, mean=mean, n_groups=N,
                )
                results[(name, k, j, m)] = (np.asarray(I_pt) - np.asarray(I_base)) * 100


In [ ]:

# --- colors (light to dark = mean 0.3, 0.5, 0.7) ---
colors_m = ['#bdc9e1', '#74a9cf', '#0570b0']
colors_t = ['#d7b5d8', '#df65b0', '#ce1256']
colors_v = ['#b2e2e2', '#66c2a4', '#238b45']

from src.contact import pol_mean_to_ab as _pmab

Lin = 178 / 25.4
fig = plt.figure(figsize=(Lin, Lin))
gs = fig.add_gridspec(2, 1, height_ratios=[0.15, 1], hspace=0.1)
gs_top = gs[0].subgridspec(1, 3, wspace=0.1)
gs_bot = gs[1].subgridspec(3, 3, hspace=0.3, wspace=0.3)

grays = ['#d9d9d9', '#969696', '#525252']

# top histograms: one row of 3 (per mean) for each column's polarization
for k in range(3):
    gs_hist = gs_top[k].subgridspec(1, 3, wspace=0.05)
    for m in range(3):
        ax_h = fig.add_subplot(gs_hist[m])
        a, b = pol_mean_to_ab(np.array(pol_vals[k]), np.array(mean_vals[m]))
        y = np.asarray(beta_populations(a, b, N))
        x = np.linspace(0, 1, N)
        ax_h.bar(x, y, width=1 / N * 0.8, color=grays[m],
                 edgecolor='black', linewidth=0.3)
        ax_h.set_xlim(-0.1, 1.1)
        ax_h.set_xticks([])
        ax_h.set_yticks([])
        if m == 1:
            ax_h.set_title(f"pol={pol_vals[k]:.2f}", fontsize=8)
        ax_h.set_xlabel(f"m={mean_vals[m]:.1f}", fontsize=8)

axs = [[fig.add_subplot(gs_bot[j, k]) for k in range(3)] for j in range(3)]

for m in range(3):
    for k in range(3):
        for j in range(3):
            M_curve = results[('M', k, j, m)]
            T_curve = results[('T', k, j, m)]
            V_curve = results[('V', k, j, m)]
            LW = 1.5
            label_kw = dict(label=None)
            if m == 2:
                axs[j][k].plot(R0s, M_curve, color=colors_m[m], linewidth=LW, linestyle='-', label='SIR-M')
                axs[j][k].plot(R0s, T_curve, color=colors_t[m], linewidth=LW, linestyle='--', label='SIR-T')
                axs[j][k].plot(R0s, V_curve, color=colors_v[m], linewidth=LW, linestyle=':', label='SIR-V')
            else:
                axs[j][k].plot(R0s, M_curve, color=colors_m[m], linewidth=LW, linestyle='-')
                axs[j][k].plot(R0s, T_curve, color=colors_t[m], linewidth=LW, linestyle='--')
                axs[j][k].plot(R0s, V_curve, color=colors_v[m], linewidth=LW, linestyle=':')
            axs[j][k].axhline(0, color='black', linestyle='--', linewidth=1)
            axs[j][k].axvline(2.5, color='black', linestyle=':', linewidth=1)
            axs[j][k].set_ylim(-50, 50)
            axs[j][k].set_xlim(1, 5)
            axs[j][k].set_xticks([1, 3, 5])
            axs[j][k].set_yticks([-50, 0, 50])
            if k == 0:
                axs[j][k].set_ylabel(f"hom={hom_vals[j]:.0f}", fontsize=10)
                axs[j][k].text(-0.25, 0.5, "Structural bias  $\\Delta I$", fontsize=8,
                               transform=axs[j][k].transAxes, ha='center', va='center', rotation=90)
            if j == 2:
                axs[j][k].set_xlabel("$R_0^*$", fontsize=10)
            axs[j][k].legend(fontsize=6)

fig.canvas.draw()

# align top histograms over their column's main plot
for k in range(3):
    hist_axes = [fig.axes[k * 3 + m] for m in range(3)]
    main_pos = axs[0][k].get_position()
    for m in range(3):
        new_x0 = main_pos.x0 + m * (main_pos.width / 3)
        old = hist_axes[m].get_position()
        hist_axes[m].set_position([new_x0, old.y0, main_pos.width / 3, old.height])

fig.savefig("figures/Fig_SI/structural_bias_comparison.pdf", dpi=300, bbox_inches='tight')